In [12]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

In [13]:
# ====================================================================== #
#  Utility                                                                #
# ====================================================================== #

def format_solutions(solutions, sep=":"):
    """
    Convert a list of solutions into readable strings.

    Each solution is a tuple/list of node names joined by `sep`.

    >>> format_solutions([("A","B","C"), ("A","C")], sep=",")
    ['A,B,C', 'A,C']
    """
    return [sep.join(str(n) for n in sol) for sol in solutions]


# ====================================================================== #
#  Graph  —  undirected, may have cycles                                  #
# ====================================================================== #

class Graph:
    """
    Undirected graph.  Edges have no direction; A-B and B-A are the same.
    May contain cycles.

    Internal state
    --------------
    _adj : dict[str, list[str]]   symmetric adjacency list
    """

    def __init__(self, edges=None, nodes=None):
        """
        edges : list of (a, b) tuples  — unordered pairs
        nodes : optional list of str   — to include isolated nodes
        """
        self._adj = {}
        if nodes:
            for n in nodes:
                self._ensure(n)
        if edges:
            for a, b in edges:
                self._ensure(a)
                self._ensure(b)
                if b not in self._adj[a]:
                    self._adj[a].append(b)
                if a not in self._adj[b]:
                    self._adj[b].append(a)

    # ── private ───────────────────────────────────────────────────────

    def _ensure(self, n):
        if n not in self._adj:
            self._adj[n] = []

    # ── accessors ─────────────────────────────────────────────────────

    def nodes(self):
        """Sorted list of all node names."""
        return sorted(self._adj)

    def edges(self):
        """List of edges as sorted (a, b) tuples — each pair appears once."""
        seen, result = set(), []
        for a in sorted(self._adj):
            for b in self._adj[a]:
                key = tuple(sorted([a, b]))
                if key not in seen:
                    seen.add(key)
                    result.append(key)
        return result

    def neighbors(self, node):
        """All nodes directly connected to node."""
        return list(self._adj.get(node, []))

    def degree(self, node):
        """Number of edges incident to node."""
        return len(self._adj.get(node, []))

    def __str__(self):
        lines = [f"{self.__class__.__name__}:"]
        for n in self.nodes():
            lines.append(f"  {n}  neighbors={sorted(self._adj[n])}")
        return "\n".join(lines)

    # ── core algorithm ────────────────────────────────────────────────

    def minimum_dominating_sets(self):
        """
        RECURSIVE — backtracking with size pruning + feasibility pruning.

        Find all minimum dominating sets S:
          every node NOT in S must be adjacent to (= one hop from) some node in S.

        Undirected coverage: coverage(u) = {u} ∪ neighbors(u)

        This is Minimum Set Cover — NP-complete in general.

        Pruning 1 — size:        if |chosen| >= best, abandon branch.
        Pruning 2 — feasibility: if some uncovered node has no remaining
                                  candidate that can cover it, abandon.

        Returns a list of optimal solutions (each a sorted list of nodes).
        """
        nodes     = self.nodes()
        n         = len(nodes)
        all_nodes = set(nodes)

        coverage    = {u: set(self.neighbors(u)) | {u} for u in nodes}
        node_idx    = {nd: i for i, nd in enumerate(nodes)}
        coverers_of = {v: set() for v in nodes}
        for u in nodes:
            for v in coverage[u]:
                coverers_of[v].add(node_idx[u])

        best_size, results = [n], []

        def backtrack(idx, chosen, covered):
            if covered == all_nodes:
                size = len(chosen)
                if size < best_size[0]:
                    best_size[0] = size
                    results.clear()
                    results.append(tuple(chosen))
                elif size == best_size[0]:
                    results.append(tuple(chosen))
                return
            if len(chosen) >= best_size[0]:               # pruning 1
                return
            for v in all_nodes - covered:                 # pruning 2
                if not any(i >= idx for i in coverers_of[v]):
                    return
            if idx >= n:
                return
            node = nodes[idx]
            chosen.append(node)                           # branch A: include
            backtrack(idx + 1, chosen, covered | coverage[node])
            chosen.pop()
            backtrack(idx + 1, chosen, covered)           # branch B: skip

        backtrack(0, [], set())
        return [sorted(r) for r in results]


## Problem 1 — Cameras


In [14]:
locaties = Graph(edges=[
        ("R", "Q"), ("U", "Z"), ("K", "Y"), ("Q", "Z"), ("L", "K")
])

solutions = locaties.minimum_dominating_sets()

print(f"Nodes              : {locaties.nodes()}")
print(f"Minimum relay size : {len(solutions[0])}")
print(f"Number of optimal solutions: {len(solutions)}")
print("All optimal relay sets:")
for s in format_solutions(solutions, sep=", "):
    print(f"  {{ {s} }}")


Nodes              : ['K', 'L', 'Q', 'R', 'U', 'Y', 'Z']
Minimum relay size : 3
Number of optimal solutions: 4
All optimal relay sets:
  { K, Q, U }
  { K, Q, Z }
  { K, R, U }
  { K, R, Z }


In [15]:
exhaustief_bepaal_locaties = locaties.minimum_dominating_sets()
print(exhaustief_bepaal_locaties)

[['K', 'Q', 'U'], ['K', 'Q', 'Z'], ['K', 'R', 'U'], ['K', 'R', 'Z']]
